# Frame reconstruction: downstream test of simulated events
Pretrained **E2VID** and **FireNet** reconstruct intensity frames from the real events and from every method's events in `data_test_evt/<DATASET>/<SPLIT>/`. Nothing is trained. All the work is done by `benchmarks/recon_eval.py`; this notebook sets it up, runs it and shows the results. The dataset is a parameter, so any folder under `data_test_evt/` works.

**Protocol (same for every event source)**
- Frame *k* (k ≥ 1) is reconstructed from the events in [t<sub>k−1</sub>, t<sub>k</sub>), so it lines up with `frames/{k:06d}.png` and `frame_ts[k]` of the source sequence.
- The protocol follows EVREAL's standard (`std`) setup: each window becomes a 5-bin voxel grid (time scaled by the window's first and last event, bilinear weights, polarity ±1, non-zero entries normalised to mean 0 and std 1), with EVREAL's centred zero padding and its model loading.
- All sources of a sequence run as one batch with one recurrent state each, so every method gets identical windows, padding and state handling. The state is reset per sequence and frames k < `WARMUP` are not scored.
- **V2V is left out**: it stores one summed voxel per frame interval, without the timing inside the interval that the networks need.

**Metrics** (outputs normalised as in EVREAL, E2VID robust 1–99 percentile and FireNet none, then clipped to [0, 1])
- `agreement_*`: reconstruction from a method's events vs reconstruction from the **real** events, same frame and network. **Main result**, independent of the GT frames.
- `gt_*`: reconstruction vs the ground-truth frame. Secondary, because the simulators were given these frames. The `real` row is the reference level.
- MSE ↓, SSIM ↑ (Gaussian 11×11, σ 1.5), LPIPS ↓ (AlexNet).

**Outputs**
- Saved frames: `data_test_recon/<DATASET>/<SPLIT>/<network>/<method>/<sequence>/{k:06d}.png`, plus `recon.json`. The method `real` is the reconstruction from real events.
- With `SAVE = "hybrid"` (default), each sequence gets 40 frames, the same indices for every method and both networks: 10 with the highest and 10 with the lowest PSNR vs GT (from the stored MSE, averaged over all methods), 10 with the largest ground-truth frame-to-frame change, and 10 spread evenly over the rest, without duplicates. Each one also gets a contact sheet in `contact_sheets/<sequence>/` (rows = networks, columns = GT, real, every method, with per-frame scores). Pointing `RUN_DIR` at a finished run reuses its metrics and only makes the frames and sheets.
- Reports: `reports/recon/<DATASET>/<SPLIT>/<timestamp>/` with `per_frame.csv`, `per_sequence.csv`, `summary.csv`, `sanity.csv`, `wilcoxon.csv`, `selection.csv`, `config.json`, the contact sheets and the qualitative figures.

**Setup is automatic.** On the first run the script clones EVREAL (github.com/ercanburak/EVREAL) at a pinned commit into `~/.cache/evtween/EVREAL`, or `EVREAL_ROOT` if set. The clone contains the model code and the authors' pretrained E2VID and FireNet weights, and both weight files are checked against pinned SHA-256 hashes. Missing packages (`lpips`, `opencv-python-headless`, `einops`) are pip-installed into the current environment, and LPIPS downloads torchvision's AlexNet weights. Add `--no-auto-setup` to fail instead of installing.

In [ ]:
from pathlib import Path
import os, sys

REPO = Path.cwd().resolve()
while not (REPO / "benchmarks" / "recon_eval.py").is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "benchmarks"))
import recon_eval

# ---- settings -------------------------------------------------------------
DATASET = "eds"            # folder name under data_test_evt/ (case-sensitive), e.g. "eds"
SPLIT = "test"
METHODS = None             # None = every method found for the dataset (V2V is always left out)
SEQUENCES = None           # None = all sequences
NETWORKS = ["e2vid", "firenet"]
WARMUP = 10
SAVE = "hybrid"            # "hybrid" | "samples" | "all" | "none"
SAMPLES = 5                # frames saved per sequence with SAVE = "samples"
DEVICES = ["cuda:0", "cuda:0"]  # one worker per entry; two workers on one A100 overlap loading and compute
RUN_DIR = None             # None = new run; "latest" = newest run of DATASET; or a report folder path

EVREAL_ROOT = None         # None = $EVREAL_ROOT or ~/.cache/evtween/EVREAL (cloned automatically if missing)

if RUN_DIR == "latest":
    runs = sorted(p for p in (REPO / "reports" / "recon" / DATASET / SPLIT).glob("*") if (p / "config.json").is_file())
    if not runs:
        raise FileNotFoundError(f"No runs under reports/recon/{DATASET}/{SPLIT}")
    RUN_DIR = str(runs[-1])
    print("latest run:", RUN_DIR)
elif RUN_DIR and not Path(RUN_DIR).is_absolute():
    RUN_DIR = str(REPO / RUN_DIR)

argv = [DATASET, "--split", SPLIT, "--networks", *NETWORKS, "--warmup", str(WARMUP),
        "--save", SAVE, "--samples", str(SAMPLES), "--devices", *DEVICES]
if METHODS: argv += ["--methods", *METHODS]
if SEQUENCES: argv += ["--sequences", *SEQUENCES]
if RUN_DIR: argv += ["--run-dir", str(RUN_DIR)]
if EVREAL_ROOT: argv += ["--evreal-root", str(EVREAL_ROOT)]
print("python benchmarks/recon_eval.py " + " ".join(argv))

## Run
Long runs are safer in a terminal (`tmux`) with the command printed above. Use the same `--run-dir` to resume: finished sequences are skipped. To only view an existing run, set `RUN_DIR` above (a folder, or `"latest"` for the newest run of `DATASET`) and skip this cell.

In [ ]:
run_dir = recon_eval.main(argv)

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image as IPImage

run_dir = Path(RUN_DIR) if RUN_DIR else run_dir
recon_eval.summarize(run_dir)
summary = pd.read_csv(run_dir / "summary.csv")
config = json.loads((run_dir / "config.json").read_text())
print(run_dir)
print("methods:", config["methods"])
print("excluded:", config["excluded"])

def table(kind):
    cols = {}
    for m in recon_eval.METRICS:
        c = f"{kind}_{m}"
        arrow = "↑" if recon_eval.HIGHER_IS_BETTER[m] else "↓"
        cols[f"{m.upper()} {arrow}"] = summary.apply(
            lambda r: f"{r[c+'_mean']:.4f} ± {r[c+'_std']:.4f} ({int(r[c+'_rank'])})"
            if pd.notna(r[c + "_mean"]) else "", axis=1)
    out = pd.concat([summary[["network", "method", "sequences", "frames"]], pd.DataFrame(cols)], axis=1)
    if kind == "agreement":
        out = out[out.method != recon_eval.REAL]
    return out.set_index(["network", "method"])

## Main result: agreement with the reconstruction from real events
Mean ± std over sequences (each sequence weighted equally), rank in brackets.

In [ ]:
display(table("agreement"))

## Secondary: against the ground-truth frames
The simulators saw these frames, so read this next to the `real` row.

In [ ]:
display(table("gt"))

## Sanity check of the inputs
Event count, rate, ON fraction and empty windows per source and sequence. `status = missing` means a method has no output for that sequence.

In [ ]:
sanity = pd.read_csv(run_dir / "sanity.csv")
display(sanity.groupby("method", sort=False)[["events", "rate_hz", "on_fraction", "empty_windows"]].mean())
display(sanity[sanity.status != "ok"])

## Best Evtween variant vs best baseline
Paired Wilcoxon signed-rank test over sequences, per network and metric. With fewer than 6 sequences the two-sided p-value cannot go below 0.05.

In [ ]:
display(pd.read_csv(run_dir / "wilcoxon.csv"))

## Qualitative comparison
The same frame reconstructed from every source, next to the ground truth. It uses the saved sample frames.

In [ ]:
sequence = config["sequences"][0]
for network in config["networks"]:
    path = recon_eval.qualitative(run_dir, sequence, network=network)
    display(IPImage(filename=str(path)))

## Contact sheets of the selected frames
`selection.csv` lists the 40 frames per sequence and why each was chosen. Pick a sequence and category to view its sheets.

In [ ]:
selection = pd.read_csv(run_dir / "selection.csv")
display(selection.groupby(["sequence", "category"]).size().unstack(fill_value=0))

SHEET_SEQUENCE = selection.sequence.iloc[0]
SHEET_CATEGORY = "low_psnr"      # "high_psnr" | "low_psnr" | "high_motion" | "temporal"
SHEET_LIMIT = 3
for path in sorted((run_dir / "contact_sheets" / SHEET_SEQUENCE).glob(f"*_{SHEET_CATEGORY}.png"))[:SHEET_LIMIT]:
    print(path.name)
    display(IPImage(filename=str(path)))